<a href="https://colab.research.google.com/github/B-MEbrahim/ECHO-Tasks/blob/main/Week2/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers faiss-cpu langchain_huggingface langchain_elasticsearch torch sentence-transformers chromadb langchain_groq

In [ ]:
!pip -q install python-dotenv

In [2]:
import faiss
import numpy as np
from elasticsearch import Elasticsearch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import os
import torch.nn.functional as F

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv('/content/.env')
hf_token = os.getenv("HF_API_TOKEN")

login(token=hf_token)

sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

In [25]:
load_dotenv('/content/.env')

True

In [ ]:
from langchain_groq import ChatGroq

# Helper function to get the language model with the working model & key
def get_llm():
    """
    Returns the language model instance.
    """
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        max_tokens=1024,
        api_key=grok_api_key # Use your working key here
    )
    return llm

# Re-initialize the model variable immediately after
chat_groq_model = get_llm()

In [11]:
from langchain_groq import ChatGroq

grok_api_key = os.getenv('GROK_API_KEY')

# Helper function to get the language model with the working model & key
def get_llm():
    """
    Returns the language model instance.
    """
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        max_tokens=1024,
        api_key=grok_api_key
    )
    return llm

# Re-initialize the model variable immediately after
chat_groq_model = get_llm()

In [12]:
from groq import Groq

# Paste your fresh key here
client = Groq(api_key=grok_api_key)

In [26]:
from elasticsearch import Elasticsearch

elastic_api_key = os.getenv('ELASTIC_API_KEY')
# Connect using your direct Elastic Cloud URL and basic_auth
es = Elasticsearch(
    hosts=["https://my-vectordb-project-a1f4a2.es.us-east4.gcp.elastic.cloud:443"],
    api_key=elastic_api_key
)

# Verify connection
if es.ping():
    print("Successfully connected to Elasticsearch!")
else:
    print("Connection failed.")

# Define the mapping for the index
index_body = {
    "mappings": {
        "properties": {
            "content": {"type": "text"}
        }
    }
}

# Create an index named 'movies'
index_name = 'movies'
if not es.indices.exists(index=index_name):
    es.indices.create(index=index_name, body=index_body)
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

Successfully connected to Elasticsearch!
Index 'movies' already exists.


In [ ]:
# Load the tokenizer for the reranking model
rerank_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load the BERT model for sequence classification (used for reranking)
rerank_model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

In [13]:
def advanced_query_transformation(query):
    """
    Transforms an input user query using Hypothetical Document Embeddings (HyDE).

    Uses Grok to generate a hypothetical passage answering the query.
    The generated text mimics the style, vocabulary, and structure of
    target documents, bridging the semantic gap during vector search.

    Args:
        query (str): The raw user query.

    Returns:
        str: A generated hypothetical document passage to be embedded.
    """
    system_prompt = (
        "You are a specialized search transformation assistant. Given a user query, "
        "write a short, factual, declarative passage (1-2 paragraphs) that directly answers "
        "the query as if it were an excerpt from a technical document or knowledge base. "
        "Do not include conversational greetings, intros, or meta-commentary."
    )
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Question: {query}\n\nWrite the answering document passage:"}
        ]
    )
    hypothetical_document = response.choices[0].message.content.strip()
    return hypothetical_document

In [14]:
def advanced_query_routing(query):
    """
    Routes a query by forcing the LLM to return a single token matching a target route.

    Args:
        query (str): The user's query.

    Returns:
        str: 'textual' if the query requires text-based retrieval, 'vector' otherwise.
    """
    prompt = f"""Classify the user query into EXACTLY ONE of these categories:
              - vector (for general explanations, concepts, and summaries)
              - textual (if query requires text-based retreval)

              User Query: "{query}"

              Output ONLY the category name (vector, textual) and nothing else."""
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    route = response.choices[0].message.content.strip().lower()

    # Simple fallback guard
    valid_routes = {"vector", "textual"}
    return route if route in valid_routes else "vector"

In [20]:
# Fusion Retrieval Function
def fusion_retrieval(query, top_k=5):
    """
    Retrieves the top_k most relevant documents using a combination of vector-based
    and textual retrieval methods.

    Args:
        query (str): The search query.
        top_k (int): The number of top documents to retrieve.

    Returns:
        list: A list of combined results from both vector and textual retrieval methods.
    """
    # Vector-based retrieval using sentence embeddings
    query_embedding = sentence_model.encode(query).tolist()
    vector_results = collection.query(query_embeddings=[query_embedding], n_results=min(top_k, len(documents)))

    # Textual retrieval using Elasticsearch
    es_body = {
        "size": top_k,  # Move size into body
        "query": {
            "match": {
                "content": query
            }
        }
    }
    es_results = es.search(index=index_name, body=es_body)
    es_documents = [hit["_source"]["content"] for hit in es_results['hits']['hits']]

    # Combine results from both retrieval methods
    combined_results = vector_results['documents'][0] + es_documents

    return combined_results


In [16]:
# Document Reranking Function
def rerank_documents(query, documents):
    """
    Reranks the retrieved documents based on their relevance to the query using a pre-trained
    BERT model.

    Args:
        query (str): The user's query.
        documents (list): A list of documents retrieved from the search.

    Returns:
        list: A list of reranked documents, sorted by relevance.
    """
    if not documents:
        return []

    scores = []
    for doc in documents:
        # Refactored: Call the tokenizer directly instead of .encode_plus()
        inputs = rerank_tokenizer(
            query,
            doc,
            return_tensors='pt',
            truncation=True,
            padding=True
        )

        # Use logits to get scores
        outputs = rerank_model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=1)
        positive_class_probability = probabilities[:, 1].item()  # Assuming the second element represents the positive class
        scores.append(positive_class_probability)

    # Sort and return documents by score in descending order
    ranked_docs = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked_docs]

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load the model and tokenizer directly
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Updated summarizer function with **kwargs to accept any generation parameters safely
def summarizer(text, max_length=130, min_length=30, **kwargs):
    """Custom summarizer function bypassing the pipeline registry."""
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

    # Extract safe parameters or use defaults
    do_sample = kwargs.get("do_sample", False)
    num_beams = 1 if do_sample else 4

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        do_sample=do_sample,
        num_beams=num_beams,
        early_stopping=True
    )
    return [{"summary_text": tokenizer.decode(summary_ids[0], skip_special_tokens=True)}]

In [18]:
# 3. Updated context compression function
def select_and_compress_context(documents):
    """
    Summarizes the content of the retrieved documents to create a compressed context.

    Args:
        documents (list): A list of documents to summarize.

    Returns:
        list: A list of summarized texts for each document.
    """
    summarized_context = []
    for doc in documents:
        input_length = len(doc.split())  # Calculate input length based on word count
        max_length = max(1, min(100, input_length))  # Ensure max_length is at least 1
        min_length = min(5, max_length)  # Adjust min_length dynamically if the doc is extremely short

        summary = summarizer(doc, max_length=max_length, min_length=min_length, do_sample=False)[0]['summary_text']
        summarized_context.append(summary)
    return summarized_context

In [19]:
# Answer Generation Function
def generate_answer(query, chunks, llm):
    """
    Generates an answer based on the input query and context chunks using a language model.

    Args:
        query (str): The user's query.
        chunks (list): A list of context chunks to inform the answer.
        llm (ChatGroq): An instance of the ChatGroq language model.

    Returns:
        str: The generated answer.
    """
    # Combine chunks into a single context string
    context = "\n\n".join(chunks)

    # Construct the prompt for the language model as a string
    prompt = f"""[INST]
Instruction: You're an expert in movie suggestions. Your task is to analyze carefully the context and come up with an exhaustive answer to the following question:
{query}

Here is the context to help you:

{context}

[/INST]"""

    # Invoke the language model with the prompt
    response = llm.invoke(prompt)  # Pass the prompt as a string directly

    # Since response is likely an AIMessage object, access the content directly
    generated_text = response.content

    return generated_text


In [21]:
def advanced_rag_pipeline(query):
    """
    The main pipeline function for the Advanced Retrieval-Augmented Generation (RAG) system.
    It processes the query, retrieves relevant documents, reranks them, selects and compresses
    the context, and finally generates an answer.

    Args:
        query (str): The user's input query.

    Returns:
        str: The final generated answer.
    """
    # Transform and route query
    transformed_query = advanced_query_transformation(query)
    retrieval_method = advanced_query_routing(transformed_query)

    # Retrieve documents using fusion retrieval
    retrieved_documents = fusion_retrieval(transformed_query)

    # Rerank documents based on relevance
    ranked_documents = rerank_documents(query, retrieved_documents)

    # Select and compress context for answer generation
    context = select_and_compress_context(ranked_documents)

    # Generate final answer based on the context
    final_answer = generate_answer(query, context, chat_groq_model)
    return final_answer

In [ ]:
import chromadb

# Initialize ChromaDB client and create collection
client = chromadb.Client()

# Define the collection name
collection_name = "movies"

try:
    # Attempt to create the collection in ChromaDB
    collection = client.create_collection(name=collection_name)
    print(f"Collection '{collection_name}' created successfully.")

    # Define the documents to be inserted into the collection
    documents = [
        {"id": "1", "content": "The Shawshank Redemption is a great movie to watch on a rainy day."},
        {"id": "2", "content": "Forrest Gump is an uplifting film perfect for a rainy afternoon."}
    ]

    # Extract the IDs and content for insertion
    ids = [doc["id"] for doc in documents]
    contents = [doc["content"] for doc in documents]

    # Insert documents into the collection
    collection.add(ids=ids, documents=contents)
    print("Documents inserted successfully.")

except Exception as e:
    print(f"Collection '{collection_name}' already exists. No need to create it again.")
    # Optionally, you could fetch the existing collection here
    collection = client.get_collection(name=collection_name)

except Exception as e:
    print(f"An error occurred: {e}")


In [24]:
# Re-initialize the Groq client to resolve the variable naming conflict with ChromaDB
from groq import Groq
client = Groq(api_key=grok_api_key)

# Example query
query = "What are some good movies to watch on a rainy day?"

# Run the query through the Advanced RAG Pipeline
answer = advanced_rag_pipeline(query)

# Output the generated answer
print(answer)

**Rain‑y‑day movie‑watching is all about finding films that match the cozy, introspective, or “let‑the‑world‑slow‑down” vibe that a drizzle outside creates.** Below is a curated list of movies (classic, modern, indie, and a few hidden gems) that work especially well when you’re curled up with a blanket, a hot drink, and the sound of rain on the window.  

---

## 1. Feel‑Good Classics (Uplifting, Warm‑hearted)

| Movie | Year | Why it fits a rainy day | Quick Synopsis |
|-------|------|------------------------|----------------|
| **Forrest Gump** | 1994 | The gentle narration, nostalgic soundtrack, and “life is like a box of chocolates” optimism make it a comforting companion while the world outside is gray. | Forrest (Tom Hanks) drifts through pivotal moments in U.S. history, sharing his simple wisdom and boundless love for Jenny. |
| **The Shawshank Redemption** | 1994 | Its themes of hope, perseverance, and eventual freedom feel especially resonant when you’re stuck indoors. | Banke

### Summary
- In query transformation I used HyDE where we transform query into a Hypothetical Document answer using a LLM.
- In query routin I used LLM Routing where we prompt the LLM to output only a single category word.